# Lab08 — *(Optional)* Publish Nova Assistant to Gemini Enterprise

**Storyline.** Nova Assistant runs, is governed and evaluated. The last step is to let employees find it where they
already work: the **Gemini Enterprise** app, the employee-facing hub for discovering and using agents.

This lab is optional because publishing needs an **active Gemini Enterprise licence assigned to you**. Without one you
can still create the app and read the two registration routes; the registration call itself answers
`FAILED_PRECONDITION: an active Gemini Enterprise license is not available`.

Run it **before** the clean-up in [Lab09](lab09_cleanup.ipynb), or keep the project around.

**You will learn**
1. What a Gemini Enterprise **app** is and how to create one in the `eu` multi-region
2. How to register an Agent Runtime agent as an **ADK agent** (one command)
3. How agents hosted anywhere are registered as **A2A agents**

Estimated time: 15 minutes.

> **Terminal or notebook — your choice.** Every cell that calls a CLI prints the exact command first (`$ …`).
> Copy it into your own terminal (from the repo root) if you prefer to run it yourself; the cells just automate the same commands.


In [ ]:
# --- Workshop configuration (same cell at the top of every lab) ---
import os, sys, json, pathlib
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID      = os.environ["PROJECT_ID"]
PROJECT_NUMBER  = os.environ["PROJECT_NUMBER"]
REGION          = os.environ["REGION"]           # europe-west1: Agent Runtime, Sessions, Memory Bank, Gateway, Model Armor
MODEL_LOCATION  = os.environ["MODEL_LOCATION"]   # eu: multi-region endpoint that serves gemini-3.8-flash
MODEL           = os.environ["MODEL"]            # gemini-3.8-flash
AGENT_NAME      = os.environ["AGENT_NAME"]       # nova-assistant
AGENT_DIR       = REPO_ROOT / AGENT_NAME

# Make every shell (!) and SDK call in this notebook target the workshop project.
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION}")
print(f"Agent dir: {AGENT_DIR}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

In [ ]:
# --- Lab08 helpers: persist ids for later, get a token for REST calls, check the deployed agent ---
import subprocess, json, requests, google.auth
from google.auth.transport.requests import Request

def save_to_workshop_env(**kv):
    """Persist values for later (workshop.env) and for this kernel."""
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    for k, v in kv.items():
        lines = [l for l in lines if not l.startswith(f"{k}=")]
        lines.append(f"{k}={v}")
        os.environ[k] = str(v)
    ENV_FILE.write_text("\n".join(lines) + "\n")
    print("saved:", ", ".join(f"{k}={v}" for k, v in kv.items()))

def gcp_token():
    """Return a short-lived access token of the notebook user for raw REST calls."""
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return creds.token

# The agent deployed in Lab02 and updated since; agents-cli publish reads its id from deployment_metadata.json.
NOVA_AGENT_ENGINE = os.environ["NOVA_AGENT_ENGINE"]
meta = json.loads((AGENT_DIR / "deployment_metadata.json").read_text())
assert meta["remote_agent_runtime_id"] == NOVA_AGENT_ENGINE, "deployment_metadata.json and workshop.env disagree - redeploy first"
print("agent to publish:", NOVA_AGENT_ENGINE)


## 8.1 What publishing needs

**Gemini Enterprise** is the employee-facing app where staff discover and use agents.
Publishing needs:

1. a Gemini Enterprise **app** (an "engine" in the API) — we create one in the **`eu`** multi-region, and
2. an **active Gemini Enterprise licence assigned to you** (a subscription on the billing account,
   distributed to this project + location, assigned to your user). The registration call checks for it —
   `FAILED_PRECONDITION: an active Gemini Enterprise license is not available` means it is not assigned yet.

With a licence, publishing is one command. Without one, read on and skip the registration cell — nothing
else depends on it.

## 8.2 Create the app (REST)

The API requires at least one data store, so we create an empty one first. (In the console:
*Gemini Enterprise → Apps → Create app*, location **eu**.)

In [ ]:
# --- Create a Gemini Enterprise app that will host the agent: data store -> app -> save id ---
# Gemini Enterprise lives in the eu multi-region (Discovery Engine API); the app needs at least one data store.
GE_LOCATION = "eu"
GE_APP_ID   = "nova-workplace"
ge = f"https://{GE_LOCATION}-discoveryengine.googleapis.com/v1/projects/{PROJECT_ID}/locations/{GE_LOCATION}/collections/default_collection"
h  = {"Authorization": f"Bearer {gcp_token()}", "Content-Type": "application/json", "X-Goog-User-Project": PROJECT_ID}

# Create an empty data store (NO_CONTENT: the app is only a shell for agents). "already exists" on re-run is fine.
r = requests.post(f"{ge}/dataStores?dataStoreId=nova-kb", headers=h, json={
        "displayName": "Nova knowledge base", "industryVertical": "GENERIC",
        "solutionTypes": ["SOLUTION_TYPE_SEARCH"], "contentConfig": "NO_CONTENT"})
print("data store:", r.status_code, r.json().get("error", {}).get("message", "ok"))

# Create the workplace app (engine) on top of it.
r = requests.post(f"{ge}/engines?engineId={GE_APP_ID}", headers=h, json={
        "displayName": "Nova Market workplace", "dataStoreIds": ["nova-kb"],
        "solutionType": "SOLUTION_TYPE_SEARCH", "industryVertical": "GENERIC",
        "appType": "APP_TYPE_INTRANET", "commonConfig": {"companyName": "Nova Market"}})
print("app       :", r.status_code, r.json().get("error", {}).get("message", "ok"))

# Save the app's full resource name (needed by agents-cli publish) and list the apps the CLI can see.
GE_APP_NAME = f"projects/{PROJECT_NUMBER}/locations/{GE_LOCATION}/collections/default_collection/engines/{GE_APP_ID}"
save_to_workshop_env(GE_APP_NAME=GE_APP_NAME)
terminal("agents-cli publish gemini-enterprise --list", cwd=AGENT_DIR)
print(subprocess.run("agents-cli publish gemini-enterprise --list", shell=True, cwd=AGENT_DIR, capture_output=True, text=True).stdout)

## 8.3 Register as an **ADK agent** (recommended for Agent Runtime)

Gemini Enterprise invokes the runtime natively (`:streamQuery`), end-to-end authenticated,
and passes the employee's email to the agent. One command, using `deployment_metadata.json`:

In [ ]:
# --- Register the deployed agent in the Gemini Enterprise app as an ADK agent ---
# Display name and description are what employees see; the tool description is what the Gemini Enterprise
# orchestrator reads to decide when to route a question to Nova. Needs a Gemini Enterprise licence for your user.
cmd = (f'agents-cli publish gemini-enterprise --registration-type adk '
       f'--gemini-enterprise-app-id {GE_APP_NAME} --display-name "Nova Assistant" '
       f'--description "Nova Market shopping and customer-care assistant: products, orders, returns." '
       f'--tool-description "Answers questions about Nova Market products, order status and return policies"')
terminal(cmd, cwd=AGENT_DIR)
r = subprocess.run(cmd, shell=True, cwd=AGENT_DIR, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])

## 8.4 Register as an **A2A agent** (for agents hosted anywhere)

The A2A route is how you would onboard an agent that does **not** run on Agent Runtime
(Cloud Run, GKE, on-prem, a partner). Gemini Enterprise fetches the **agent card** and
speaks JSON-RPC to it. Two things to know:

* Gemini Enterprise must be able to reach the card URL. For a Cloud Run service, grant
  `roles/run.invoker` to the Discovery Engine service agent
  `service-PROJECT_NUMBER@gcp-sa-discoveryengine.iam.gserviceaccount.com`.
* Traffic to A2A agents registered this way does **not** pass through Agent Gateway.

You can deploy the very same project to Cloud Run with the CLI (`--deployment-target cloud_run`
overrides the manifest) and register its card:

```bash
cd nova-assistant
agents-cli deploy --deployment-target cloud_run --project $PROJECT_ID --region $REGION --no-confirm-project
gcloud run services add-iam-policy-binding nova-assistant --region $REGION \
    --member=serviceAccount:service-$PROJECT_NUMBER@gcp-sa-discoveryengine.iam.gserviceaccount.com --role=roles/run.invoker
agents-cli publish gemini-enterprise --registration-type a2a \
    --agent-card-url https://nova-assistant-$PROJECT_NUMBER.$REGION.run.app/a2a/app/.well-known/agent-card.json \
    --gemini-enterprise-app-id $GE_APP_NAME
```

Agent Runtime stays the home of Nova Assistant, so this route is a documented alternative rather than executed here.

## Recap

* A Gemini Enterprise **app** is an engine in the `eu` multi-region with at least one data store; creating it needs no licence.
* **ADK registration** is native for Agent Runtime agents: one `agents-cli publish gemini-enterprise` command, end-to-end authenticated, with the employee's identity passed to the agent.
* **A2A registration** is the universal route for agents hosted anywhere; traffic to those agents does not pass through Agent Gateway.
* Both need an active Gemini Enterprise licence for the person who registers.

When you are done, clean up in [Lab09](lab09_cleanup.ipynb).
